# 02 — Rapport individuel par GHU

Ce notebook génère un rapport HTML pour un GHU spécifique.

Modifiez le paramètre `GHU_NAME` pour changer de groupe.

**Sortie** : `output/rapport_<ghu>.html`

In [ ]:
# Paramètre papermill — modifiable
GHU_NAME = 'GHU Nord'

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))

from pathlib import Path
DATA_DIR   = Path('../data')
OUTPUT_DIR = Path('../output')
OUTPUT_DIR.mkdir(exist_ok=True)

import pandas as pd

aphp = pd.read_csv(DATA_DIR / 'aphp_data.csv')
reg  = pd.read_csv(DATA_DIR / 'regional_data.csv')

GHU_LIST = ['GHU Centre', 'GHU Mondor', 'GHU Nord', 'GHU PSSD', 'GHU PSL', 'GHU SUN']
assert GHU_NAME in GHU_LIST, f"{GHU_NAME} non reconnu. Options : {GHU_LIST}"

ghu_data   = aphp[(aphp.entite == GHU_NAME) & (aphp.appareil == 'TOTAL')]
aphp_total = aphp[(aphp.entite == 'AP-HP') & (aphp.appareil == 'TOTAL')]
print(f"GHU sélectionné : {GHU_NAME}")
ghu_data

## Visualisations

In [ ]:
from chart_utils import (
    line_evolution, donut_market_share, heatmap_appareils,
    regional_comparison, TREATMENT_COLS, GHU_LIST
)

# Évolution patients vs AP-HP
compare = aphp[(aphp.entite.isin([GHU_NAME, 'AP-HP'])) & (aphp.appareil == 'TOTAL')]
fig = line_evolution(compare, 'annee', 'nb_patients', 'entite',
                     f'Patients — {GHU_NAME} vs AP-HP',
                     entities=[GHU_NAME, 'AP-HP'])
fig.show()

In [ ]:
# Part de marché relative
import pandas as pd

share = ghu_data.copy().reset_index(drop=True)
aphp_ref = aphp_total.copy().reset_index(drop=True)
share = share.sort_values('annee').reset_index(drop=True)
aphp_ref = aphp_ref.sort_values('annee').reset_index(drop=True)
share['part_marche'] = share['nb_patients'].values / aphp_ref['nb_patients'].values * 100

fig_s = line_evolution(share, 'annee', 'part_marche', 'entite',
                       f'Part de marché dans l\'AP-HP — {GHU_NAME} (%)',
                       entities=[GHU_NAME])
fig_s.show()

In [ ]:
# Séjours par type
melted = ghu_data.melt(
    id_vars=['annee'], value_vars=list(TREATMENT_COLS.keys()),
    var_name='type', value_name='nb_sejours')
melted['label'] = melted['type'].map(TREATMENT_COLS)

fig_t = line_evolution(melted, 'annee', 'nb_sejours', 'label',
                       f'Séjours par mode de PEC — {GHU_NAME}',
                       entities=list(TREATMENT_COLS.values()))
fig_t.show()

In [ ]:
# Heatmap par appareil
fig_h = heatmap_appareils(aphp, GHU_NAME)
fig_h.show()

## Génération HTML

In [ ]:
from report_builder import build_rapport_ghu

out = build_rapport_ghu(GHU_NAME, DATA_DIR, OUTPUT_DIR)
print(f"Rapport généré : {out}")